<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# Проект «Анализ вакансий из HeadHunter»
   

In [58]:
import pandas as pd
import psycopg2
import warnings

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable"
)

In [ ]:
# вставьте сюда параметры подключения из юнита 1. Работа с базой данных из Python


In [60]:
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)

## 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в базе (вакансии находятся в таблице `vacancies`).

In [61]:
# текст запроса
query_3_1 = f'''select count(*) as vacancies_count  -- считает количество строк, каждая строка = одна вакансия
                from vacancies                      -- используем таблицу, в которой хранятся все вакансии
            '''

In [62]:
# результат запроса
df = pd.read_sql_query(query_3_1, connection)
df

,vacancies_count
0,49197


2. Напишите запрос, который посчитает количество работодателей (таблица `employers`).

In [63]:
# текст запроса
query_3_2 = f'''select count(*) as employers_count  -- считает все строки таблицы, каждая строка = один работодатель
                from employers                      -- берём данные из справочника работодателей
            '''

In [64]:
# результат запроса
df = pd.read_sql_query(query_3_2, connection)
df

,employers_count
0,23501


3. Посчитайте с помощью запроса количество регионов (таблица `areas`).

In [65]:
# текст запроса
query_3_3 = f'''select count(*) as areas_count      -- считает количество строк, каждая строка соответствует одному региону
                from areas                          -- берём данные из справочника регионов
            '''

In [66]:
# результат запроса
df = pd.read_sql_query(query_3_3, connection)
df

,areas_count
0,1362


4. Посчитайте с помощью запроса количество сфер деятельности в базе (таблица `industries`).

In [67]:
# текст запроса
query_3_4 = f'''select count(*) as industries_count  -- считает количество строк, каждая строка соответствует одной сфере деятельности
                from industries                      -- справочник сфер деятельности
            '''

In [68]:
# результат запроса
df = pd.read_sql_query(query_3_4, connection)
df

,industries_count
0,294


***

### Выводы по предварительному анализу данных

В базе данных содержится 49 197 вакансий, что является достаточным объёмом для проведения анализа рынка труда и построения модели рекомендаций.  
Количество работодателей составляет 2 3501, что указывает на разнообразие компаний, представленных в выборке.  

В базе представлено 1 362 региона, следовательно, данные охватывают широкий географический диапазон. Это позволяет учитывать фактор локации при анализе вакансий и формировании рекомендаций.  

Также в базе выделено 294 сферы деятельности, что говорит о высокой предметной вариативности вакансий и работодателей.

## 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (`cnt`) вакансий в каждом регионе (`area`).
Отсортируйте по количеству вакансий в порядке убывания.

In [69]:
# текст запроса
query_4_1 = f'''select a.name as area,              -- берем название региона
                       count(v.id) as cnt           -- считаем количество вакансий в каждом регионе
                from vacancies v                    -- из таблицы с вакансиями
                join areas a on v.area_id = a.id    -- связываем вакансии с регионами по id региона
                group by a.name                     -- группируем вакансии по региону
                order by cnt desc                   -- сортируем регионы по убыванию количества вакансий
            '''

In [70]:
# результат запроса
df = pd.read_sql_query(query_4_1, connection)
df

,area,cnt
0,Москва,5333
1,Санкт-Петербург,2851
2,Минск,2112
3,Новосибирск,2006
4,Алматы,1892
...,...,...
764,Тарко-Сале,1
765,Новоаннинский,1
766,Бирск,1
767,Сасово,1


2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [71]:
# текст запроса
query_4_2 = f'''select count(*) as cnt              -- считаем количество вакансий
                from vacancies                      -- из таблицы с вакансиями
                where salary_from is not null       -- оставляем вакансии, где указана либо нижняя граница зп
                   or salary_to is not null         -- либо верхняя
            '''

In [72]:
# результат запроса
df = pd.read_sql_query(query_4_2, connection)
df

,cnt
0,24073


3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до **целого числа**.

In [73]:
# текст запроса
query_4_3 = f'''select round(avg(salary_from)) as avg_salary_from,   -- считаем среднее значение нижней границы зп и округляем до целого
                       round(avg(salary_to)) as avg_salary_to        -- считаем среднее значение верхней границы зп и округляем до целого
                from vacancies                                       -- из таблицы с вакансиями
                where salary_from is not null                        -- оставляем вакансии с заполненной нижней границей
                   or salary_to is not null                          -- либо с заполненной верхней границей зарплаты
            '''

In [74]:
# результат запроса
df = pd.read_sql_query(query_4_3, connection)
df

,avg_salary_from,avg_salary_to
0,71065.0,110537.0


4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (`schedule`) и типа трудоустройства (`employment`), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [75]:
# текст запроса
query_4_4 = f'''select schedule,                     -- тип рабочего графика
                       employment,                   -- тип трудоустройства
                       count(*) as cnt               -- считаем количество вакансий для каждой комбинации
                from vacancies                       -- из таблицы с вакансиями
                group by schedule, employment        -- группируем по сочетанию графика и типа трудоустройства
                order by cnt desc                    -- сортируем по убыванию вакансий
            '''

In [76]:
# результат запроса
df = pd.read_sql_query(query_4_4, connection)
df

,schedule,employment,cnt
0,Полный день,Полная занятость,35367
1,Удаленная работа,Полная занятость,7802
2,Гибкий график,Полная занятость,1593
3,Удаленная работа,Частичная занятость,1312
4,Сменный график,Полная занятость,940
5,Полный день,Стажировка,569
6,Вахтовый метод,Полная занятость,367
7,Полный день,Частичная занятость,347
8,Гибкий график,Частичная занятость,312
9,Полный день,Проектная работа,141


5. Напишите запрос, выводящий значения поля «Требуемый опыт работы» (`experience`) в порядке возрастания количества вакансий, в которых указан данный вариант опыта.

In [77]:
query_4_5 = f'''select experience,         -- вариант опыта работы
                       count(*) as cnt     -- считаем количество вакансий для каждого варианта опыта
                from vacancies             -- из таблицы с вакансиями
                group by experience        -- группируем вакансии по опыту
                order by cnt asc           -- сортируем по возрастанию
            '''

In [78]:
# результат запроса
df = pd.read_sql_query(query_4_5, connection)
df

,experience,cnt
0,Более 6 лет,1337
1,Нет опыта,7197
2,От 3 до 6 лет,14511
3,От 1 года до 3 лет,26152


***

### Выводы по детальному анализу вакансий

Наибольшее количество вакансий сосредоточено в крупных городах. Лидером является Москва**, за ней следуют Санкт-Петербург** и Минск, что отражает концентрацию работодателей и IT-рынка в крупнейших экономических центрах.

Информация о заработной плате указана хотя бы в одном из полей у 24 073 вакансий, то есть примерно у половины всех предложений. Это необходимо учитывать при дальнейшем анализе зарплат и построении рекомендаций.

Среднее значение нижней границы зарплатной вилки составляет 71 065, а верхней — 110 537, что позволяет оценить общий уровень доходов, предлагаемых на рынке труда.

Наиболее распространённым сочетанием условий работы является полный рабочий день и полная занятость. Также заметную долю занимают вакансии с удалённым форматом работы, преимущественно при полной занятости, что говорит о высокой востребованности удалённой работы.

По требуемому опыту работы наибольшее количество вакансий ориентировано на специалистов с опытом от 1 года до 3 лет, а наименьшее — на кандидатов с опытом более 6 лет. Это указывает на высокий спрос на специалистов начального и среднего уровня по сравнению с высокоэкспертными позициями.

## 5. Анализ работодателей

1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [79]:
# текст запроса
query_5_1 = f'''select name,                                      -- название работодателя
                       cnt                                        -- количество вакансий у работодателя
                from (
                        select e.name,                            -- имя работодателя
                               count(v.id) as cnt,                -- количество вакансий у работодателя
                                -- нумеруем строки после группировки по убыванию количества вакансий
                               row_number() over (order by count(v.id) desc) as rn  
                        from employers e
                        join vacancies v on e.id = v.employer_id  -- связываем вакансии с работодателями
                        group by e.name                           -- группируем по работодателю
                    ) t
                where rn in (1, 5)                                -- выбираем 1 и 5 места в рейтинге
                order by rn                                       -- упорядочиваем результат по месту
            '''

In [80]:
# результат запроса
df = pd.read_sql_query(query_5_1, connection)
df

,name,cnt
0,Яндекс,1933
1,Газпром нефть,331


2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [81]:
# текст запроса
query_5_2 = f'''select a.name as area,                                 -- название региона
                       coalesce(e.employers_cnt, 0) as employers_cnt,  -- количество работодателей в регионе, null заменяем на 0
                       coalesce(v.vacancies_cnt, 0) as vacancies_cnt   -- количество вакансий в регионе, null заменяем на 0
                from areas a                                           -- таблица регионов
                left join (
                        select area,                                 -- идентификатор региона регистрации работодателя
                               count(*) as employers_cnt             -- считаем работодателей в каждом регионе
                        from employers
                        group by area                                -- группируем работодателей по региону
                ) e on e.area = a.id                                 -- присоединяем количество работодателей к регионам
                left join (
                        select area_id,                               -- идентификатор региона вакансии
                               count(*) as vacancies_cnt              -- считаем вакансии в каждом регионе
                        from vacancies
                        group by area_id                              -- группируем вакансии по региону
                ) v on v.area_id = a.id                               -- присоединяем количество вакансий к регионам
                order by vacancies_cnt asc,                           -- сначала регионы без вакансий
                         employers_cnt desc                           -- среди них выбираем регион с максимумом работодателей
            '''

In [82]:
# результат запроса
df = pd.read_sql_query(query_5_2, connection)
df

,area,employers_cnt,vacancies_cnt
0,Россия,410,0
1,Казахстан,207,0
2,Московская область,75,0
3,Краснодарский край,19,0
4,Ростовская область,18,0
...,...,...,...
1357,Алматы,721,1892
1358,Новосибирск,573,2006
1359,Минск,1115,2112
1360,Санкт-Петербург,2217,2851


3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [83]:
# текст запроса
query_5_3 = f'''select e.name,                                    -- каждая строка результата соответствует одному работодателю
                       count(distinct v.area_id) as areas_cnt     -- считаем количество уникальных регионов, где работодатель публиковал вакансии
                from employers e                                  -- из списка работодателей
                join vacancies v on e.id = v.employer_id          -- оставляем только работодателей, у которых есть вакансии
                group by e.name                                   -- группируем работодателю
                order by areas_cnt desc                           -- сортируем работодателей по убыванию количества регионов
            '''

In [84]:
# результат запроса
df = pd.read_sql_query(query_5_3, connection)
df

,name,areas_cnt
0,Яндекс,181
1,Ростелеком,152
2,Спецремонт,116
3,Поляков Денис Иванович,88
4,ООО ЕФИН,71
...,...,...
14761,UniSol,1
14762,UNISTORY LLC,1
14763,UNIT6,1
14764,United Distribution,1


4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности.

In [85]:
# текст запроса
query_5_4 = f'''select count(*) as employers_cnt       -- считаем работодателей без сфер деятельности
                from employers e                       -- берём всех работодателей
                left join employers_industries ei 
                       on e.id = ei.employer_id        -- присоединяем сферы деятельности, если они есть
                where ei.industry_id is null           -- оставляем работодателей, у которых нет ни одной записи о сфере
            '''

In [86]:
# результат запроса
df = pd.read_sql_query(query_5_4, connection)
df

,employers_cnt
0,8419


5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности.

In [87]:
# текст запроса
query_5_5 = f'''select e.name                          -- название компании
                from employers e                       -- из таблицы работодателей
                join employers_industries ei 
                     on e.id = ei.employer_id          -- оставляем работодателей с указанными сферами деятельности
                group by e.id, e.name                  -- агрегируем строки по работодателю
                having count(ei.industry_id) = 4       -- отбираем компании ровно с четырьмя сферами деятельности
                order by e.name                        -- сортируем компании в алфавитном порядке
                offset 2 limit 1                       -- пропускаем первые две компании и берём третью
            '''

In [88]:
# результат запроса
df = pd.read_sql_query(query_5_5, connection)
df

,name
0,2ГИС


6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана «Разработка программного обеспечения».


In [89]:
# текст запроса
query_5_6 = f'''select count(distinct e.id) as employers_cnt          -- количество уникальных работодателей
                from employers e                                      -- из таблицы работодателей
                join employers_industries ei 
                     on e.id = ei.employer_id                         -- связываем работодателей с их сферами деятельности
                join industries i 
                     on ei.industry_id = i.id                         -- получаем названия сфер деятельности
                where i.name = 'Разработка программного обеспечения'  -- оставляем работодателей с нужной сферой
            '''

In [90]:
# результат запроса
df = pd.read_sql_query(query_5_6, connection)
df

,employers_cnt
0,3553


7. Для компании «Яндекс» выведите список [городов-миллионников](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8), в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку "Total" с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

    Если возникнут трудности с этим заданием, посмотрите материалы модуля 6.4 «Как получать данные из веб-источников и API».

In [91]:
# код для получения списка городов-милионников
import requests

# Получаем список городов-миллионников из Википедии через API

url = "https://ru.wikipedia.org/w/api.php"
headers = {"User-Agent": "Mozilla/5.0 (DataScienceProject/1.0)"}

cities = []
cmcontinue = None
# Используем параметр cmcontinue и цикл, пока не закончатся страницы
while True:
    params = {
        "action": "query",
        "format": "json",
        "list": "categorymembers",
        "cmtitle": "Категория:Города-миллионеры России",
        "cmnamespace": 0, 
        "cmlimit": 500,
        "origin": "*"
    }
    if cmcontinue:
        params["cmcontinue"] = cmcontinue # запрашиваем следующую страницу результатов

    r = requests.get(url, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json()

    # добавляем названия городов с текущей страницы
    cities.extend([x["title"] for x in data["query"]["categorymembers"]])

    # если продолжения нет, значит страницы кончились
    if "continue" not in data:
        break
    cmcontinue = data["continue"]["cmcontinue"]
# Результат приводим к tuple уникальных названий городов, чтобы подставить в SQL 
cities_tuple = tuple(
    city for city in set(cities) # удаляем дубликаты
    if city != "Города-миллионеры России" # исключаем лишнюю страницу категории
)

cities_tuple

('Краснодар',
 'Казань',
 'Нижний Новгород',
 'Москва',
 'Омск',
 'Санкт-Петербург',
 'Воронеж',
 'Уфа',
 'Пермь',
 'Самара',
 'Новосибирск',
 'Волгоград',
 'Красноярск',
 'Ростов-на-Дону',
 'Екатеринбург',
 'Челябинск')

In [92]:
# текст запроса
query_5_7 = f'''select a.name as area,                             -- название города
                       count(v.id) as vacancies_cnt                -- считаем количество вакансий Яндекса в городе
                from vacancies v                                   -- из таблицы с вакансиями
                join employers e on v.employer_id = e.id           -- связываем вакансии с работодателями
                join areas a on v.area_id = a.id                   -- связываем вакансии с регионами
                where e.name = 'Яндекс'                            -- оставляем вакансии Яндекса
                  and a.name in {cities_tuple}                     -- ограничиваемся городами-миллионниками
                group by a.name                                    -- группируем вакансии по городу

                union all  -- добавляем строку с общим количеством вакансий компании по cities_tuple

                select 'Total' as area,                            -- строка с общим итогом по всем городам
                       count(v.id) as vacancies_cnt                -- считаем общее количество вакансий Яндекса
                from vacancies v                                   
                join employers e on v.employer_id = e.id           -- связываем вакансии с работодателями
                join areas a on v.area_id = a.id                   -- связываем вакансии с регионами
                where e.name = 'Яндекс'                            -- оставляем вакансии Яндекса
                  and a.name in {cities_tuple}                     -- учитываем только cities_tuple

                order by vacancies_cnt asc                         -- сортируем результат по возрастанию
            '''

In [93]:
# результат запроса
df = pd.read_sql_query(query_5_7, connection)
df

,area,vacancies_cnt
0,Омск,21
1,Челябинск,22
2,Красноярск,23
3,Волгоград,24
4,Пермь,25
5,Казань,25
6,Ростов-на-Дону,25
7,Уфа,26
8,Самара,26
9,Краснодар,30


***

### Выводы по анализу работодателей

Наибольшее количество вакансий в базе размещает компания Яндекс. 

Анализ распределения работодателей и вакансий по регионам показывает, что в ряде регионов присутствуют работодатели, но при этом вакансии отсутствуют. В то же время максимальное количество работодателей и вакансий сосредоточено в крупнейших городах, прежде всего в Москве и Санкт-Петербурге, что отражает высокую концентрацию бизнеса и IT-рынка в данных регионах.

Компания Яндекс представлена в наибольшем числе регионов (181 регион), что говорит о широкой географии присутствия и масштабности бизнеса. Другие работодатели значительно уступают по этому показателю.

Существенная часть работодателей (8419 компаний) не указала сферу своей деятельности, что может негативно повлиять на точность тематического анализа и рекомендаций и требует учёта при дальнейшей обработке данных.

Количество работодателей, относящихся к сфере Разработка программного обеспечения», составляет 3553, что подтверждает значительную долю IT-компаний в базе.

В городах-миллионниках компания Яндекс разместила 485 вакансий. Наибольшее число вакансий сосредоточено в Москве и Санкт-Петербурге, тогда как в других крупных городах количество предложений ниже. Это подчёркивает важность фактора локации при построении рекомендательной модели.

## 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

    Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова `'data'` или `'данн'`.

    *Обратите внимание, что названия вакансий могут быть написаны в любом регистре.*


In [94]:
# текст запроса
query_6_1 = f'''select count(*) as vacancies_cnt       -- количество вакансий, удовлетворяющих условию
                from vacancies                         -- из таблицы с вакансиями
                where lower(name) like '%data%'        -- отбираем вакансии, в названии которых есть 'data' в любом регистре
                   or lower(name) like '%данн%'        -- либо присутствует 'данн' 
            '''

In [95]:
# результат запроса
df = pd.read_sql_query(query_6_1, connection)
df

,vacancies_cnt
0,1771


2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
    * 'data scientist'
    * 'data science'
    * 'исследователь данных'
    * 'ML' (здесь не нужно брать вакансии по HTML)
    * 'machine learning'
    * 'машинн%обучен%'

    **В следующих заданиях мы продолжим работать с вакансиями по этому условию.**

    Считаем вакансиями для специалистов уровня Junior следующие:
    + в названии есть слово “junior” **или**
    + требуемый опыт — «Нет опыта» **или**
    + тип трудоустройства — «Стажировка».


In [96]:
# текст запроса
# набор всех data science / ML вакансий
ds_cte = f'''with ds_vacancies as (
                 select *
                 from vacancies
                 where (
                         lower(name) like '%data scientist%'
                      or lower(name) like '%data science%'
                      or lower(name) like '%исследователь данных%'
                      or (
                             lower(name) like '%ml%'
                         and lower(name) not like '%html%'
                         )
                      or lower(name) like '%machine learning%'
                      or lower(name) like '%машинн%обучен%'
                       )
             )
         '''
# из ds_cte отбираются вакансии, подходящие для джунов
query_6_2 = f'''{ds_cte}
                select count(*) as vacancies_cnt                  -- считаем количество вакансий для джунов
                from ds_vacancies                                 -- только DS вакансии
                where (
                        lower(name) like '%junior%'               -- вакансии с junior
                     or experience = 'Нет опыта'                  -- либо нет опыта
                     or employment = 'Стажировка'                 -- либо стажировка
                      )
            '''

In [97]:
# результат запроса
df = pd.read_sql_query(query_6_2, connection)
df

,vacancies_cnt
0,51


3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или Postgres?

    *Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [98]:
# текст запроса
# из ds_cte отбираются вакансии, c подходящими условиями
query_6_3 = f'''{ds_cte}
                select count(*) as vacancies_cnt                  -- количество DS-вакансий с фильтрами
                from ds_vacancies                                 -- только DS вакансии
                where (
                        lower(key_skills) like '%sql%'            -- вакансии  c SQL
                     or lower(key_skills) like '%postgres%'       -- вакансии с Postgres
                     or lower(key_skills) like '%postgresql%'     -- вакансии с PostgreSQL
                      )
            '''

In [99]:
# результат запроса
df = pd.read_sql_query(query_6_3, connection)
df

,vacancies_cnt
0,229


4. Проверьте, насколько популярен Python в требованиях работодателей к DS. Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

    *Это можно сделать помощью запроса, аналогичного предыдущему.*

In [100]:
# текст запроса
query_6_4 = f'''{ds_cte}
                select count(*) as vacancies_cnt                  -- считаем DS-вакансии с Python
                from ds_vacancies                                 -- только DS вакансии
                where lower(key_skills) like '%python%'           -- отбираем вакансии с Python
            '''

In [101]:
# результат запроса
df = pd.read_sql_query(query_6_4, connection)
df

,vacancies_cnt
0,357


5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до **двух знаков после точки-разделителя**.

In [102]:
# Посмотрим какой разделитель используется
query = f'''select key_skills
from vacancies
where key_skills is not null
limit 3;
         '''

df = pd.read_sql_query(query, connection)
df

,key_skills
0,Пользователь ПК\tРабота в команде\tРемонт ноут...
1,Средства криптографической защиты информации\t...
2,Spring Framework\tSQL\tHibernate ORM\tJava\tGit


In [103]:
# текст запроса
query_6_5 = f'''{ds_cte}
                select round(avg(skills_cnt), 2) as avg_skills_cnt  -- считаем среднее количество навыков и округляем до двух знаков
                from (
                        select case
                                   when key_skills is null 
                                     or key_skills = '' then 0      -- если навыки не указаны, считаем 0
                                   else length(key_skills)          
                                      - length(replace(key_skills, chr(9), '')) + 1 -- иначе по числу табуляций между ними
                                      
                               end as skills_cnt
                        from ds_vacancies                           -- используем только DS-вакансии
                     ) t
            '''

In [104]:
# результат запроса
df = pd.read_sql_query(query_6_5, connection)
df

,avg_skills_cnt
0,5.88


6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в среднем указывают для каждого типа требуемого опыта (уникальное значение из поля `experience`).

    При решении задачи примите во внимание следующее:
    1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
    2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
    3. Если в расчётах участвует `null`, в результате он тоже даст `null` (посмотрите, что возвращает запрос `select 1 + null`). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит `null` на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

    Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до **целого числа**.

In [105]:
# текст запроса
query_6_6 = f'''{ds_cte}
                select round(avg(salary), 0) as avg_salary                   -- считаем среднюю зарплату по вакансиям и округляем до целого
                from (
                        select case
                                   when salary_from is not null 
                                    and salary_to is not null
                                       then (salary_from + salary_to) / 2.0  -- если обе границы заданы, берём среднее по вилке
                                   else coalesce(salary_from, salary_to)     -- если задана одна граница, берём её
                               end as salary
                        from ds_vacancies                                     -- используем только DS-вакансии
                        where (salary_from is not null 
                            or salary_to is not null)                         -- берём только вакансии, где заполнена хотя бы одна граница зарплаты
                          and experience = 'От 3 до 6 лет'                    -- оставляем вакансии с опытом 3–6 лет
                     ) t
            '''

In [106]:
# результат запроса
df = pd.read_sql_query(query_6_6, connection)
df

,avg_salary
0,256454.0


***

### Выводы по предметному анализу

 * В базе данных выявлено 1 771 вакансия, имеющая отношение к работе с данными, что показывает заметную представленность data-направления на рынке труда.

* Количество вакансий для начинающих составляет всего 51, что свидетельствует о высоком пороге входа в профессию и ограниченном числе предложений для специалистов без опыта или уровня Junior.

* Среди вакансий для Data Scientists 229 содержат в требованиях знание SQL, а 357 вакансий указывают Python в качестве ключевого навыка. Это подтверждает, что Python является наиболее востребованным инструментом для специалистов по данным, при этом SQL также остаётся важным, но менее универсальным требованием.

* В среднем в вакансиях для Data Scientists указывается около 6 ключевых навыков, что говорит о высоких требованиях к широте технической экспертизы кандидатов.

* Средняя зарплата для дата-сайентистов с опытом работы от 3 до 6 лет составляет около 256 454, что отражает высокий уровень дохода специалистов среднего уровня и подчёркивает экономическую привлекательность профессии.

* В совокупности результаты предметного анализа показывают, что рынок вакансий Data Scientist ориентирован преимущественно на специалистов с опытом, предъявляет высокие требования к набору навыков и предлагает конкурентный уровень заработной платы.

## Общий вывод по проекту

# подведем итог исследования, обобщите выводы

### Итоговые выводы по исследованию данных

База данных содержит 49 197 вакансий, размещённых 23 501 работодателем в 1 362 регионах и относящихся к 294 сферам деятельности. Такой объём и разнообразие данных позволяют получить репрезентативное представление о рынке труда и использовать результаты анализа для дальнейшего построения моделей рекомендаций.

Распределение вакансий по регионам является неравномерным: наибольшее количество предложений сосредоточено в крупнейших городах, прежде всего в Москве и Санкт-Петербурге а также в других крупных экономических центрах. При этом в ряде регионов присутствуют работодатели, но отсутствуют активные вакансии, что указывает на географическую асимметрию рынка труда.

Информация о заработной плате указана примерно у половины вакансий, что необходимо учитывать при дальнейшем анализе доходов. Наиболее распространённым форматом работы остаётся полный рабочий день и полная занятость однако удалённый формат также занимает значительную долю, отражая современные тенденции в IT-сфере.

Анализ работодателей показал, что рынок характеризуется высокой фрагментированностью: большинство компаний размещают ограниченное число вакансий, тогда как основная часть предложений формируется несколькими крупными работодателями. Яндекс является лидером по количеству вакансий и географии присутствия, размещая вакансии более чем в 180 регионах.

Значительная часть работодателей (8 419 компаний) не указала сферу своей деятельности, что является ограничением качества данных и может снижать точность тематического анализа и рекомендаций. При этом 3 553 работодателя относятся к сфере «Разработка программного обеспечения», что подтверждает доминирующую роль IT-компаний в выборке.

Вакансии, связанные с анализом данных, составляют 1 771 предложение, однако только 51 вакансия ориентирована на специалистов уровня Junior. Это свидетельствует о высоком пороге входа в профессию Data Scientist и ориентации рынка на специалистов с опытом работы.

Ключевыми навыками для Data Scientist являются Python и SQL, при этом в среднем в одной вакансии указывается около 6 ключевых навыков что подчёркивает высокий уровень требований к профессиональной подготовке кандидатов.

Средний уровень заработной платы для дата-сайентистов с опытом работы от 3 до 6 лет составляет около 256 454, что отражает высокую ценность специалистов данного уровня и подтверждает экономическую привлекательность профессии.

В целом результаты исследования показывают, что рынок вакансий Data Scientist является конкурентным, территориально концентрированным и ориентированным на квалифицированных специалистов.

## Дополнительные исследования данных, прогнозы, варианты продолжения исследования

In [107]:
# CTE с вакансиями Data Scientist
# Используется во всех последующих запросах

### Распределение зарплат DS по регионам

In [108]:
# текст запроса
query_salary_by_area = f'''{ds_cte}
                               select a.name as area,                            -- название региона
                                      count(*) as vacancies_cnt,                 -- количество DS-вакансий
                                      round(avg(salary), 0) as avg_salary,       -- средняя зарплата
                                      percentile_cont(0.5)
                                          within group (order by salary)
                                          as med_salary                           -- медианная зарплата
                               from (
                                       select area_id,
                                              case
                                                  when salary_from is not null
                                                   and salary_to is not null
                                                      then (salary_from + salary_to) / 2.0 -- среднее по вилке
                                                  else coalesce(salary_from, salary_to)    -- если указана одна граница
                                              end as salary
                                       from ds_vacancies                           -- только DS-вакансии
                                       where salary_from is not null
                                          or salary_to is not null               -- берём вакансии с зарплатой
                                    ) s
                               join areas a on s.area_id = a.id                   -- добавляем название региона
                               group by a.name
                               order by avg_salary desc
                            '''

In [109]:
# результат запроса
df = pd.read_sql_query(query_salary_by_area, connection)
df

,area,vacancies_cnt,avg_salary,med_salary
0,Калининград,1,425000.0,425000.0
1,Тбилиси,1,425000.0,425000.0
2,Минск,2,314785.0,314785.0
3,Кипр,1,300000.0,300000.0
4,Армения,3,268863.0,292242.0
5,Ростов-на-Дону,3,241667.0,250000.0
6,Черногория,1,233794.0,233794.0
7,Сербия,1,233794.0,233794.0
8,Турция,1,233794.0,233794.0
9,Москва,29,214000.0,200000.0


 ### Сравнение зарплат: junior / middle / senior

In [110]:
# текст запроса
query_salary_by_level = f'''{ds_cte}
                                select level,                                    -- уровень специалиста
                                       count(*) as vacancies_cnt,                -- количество вакансий
                                       round(avg(salary), 0) as avg_salary,      -- средняя зарплата
                                       percentile_cont(0.5)
                                           within group (order by salary)
                                           as med_salary                          -- медианная зарплата
                                from (
                                        select case
                                                   when experience in ('Нет опыта', 'От 1 года до 3 лет')
                                                       then 'junior'
                                                   when experience = 'От 3 до 6 лет'
                                                       then 'middle'
                                                   when experience = 'Более 6 лет'
                                                       then 'senior'
                                               end as level,                      -- классификация по опыту
                                               case
                                                   when salary_from is not null
                                                    and salary_to is not null
                                                       then (salary_from + salary_to) / 2.0
                                                   else coalesce(salary_from, salary_to)
                                               end as salary
                                        from ds_vacancies
                                        where salary_from is not null
                                           or salary_to is not null               -- учитываем только вакансии с зарплатой
                                     ) t
                                where level is not null
                                group by level
                                order by avg_salary desc
                             '''

In [111]:
# результат запроса
df = pd.read_sql_query(query_salary_by_level, connection)
df

,level,vacancies_cnt,avg_salary,med_salary
0,middle,41,256454.0,233794.0
1,senior,4,157933.0,170866.0
2,junior,39,131743.0,120000.0


### Топ-10 самых частых навыков DS

In [112]:
# текст запроса
query_top_skills = f'''{ds_cte}
                           select skill,                                        -- название навыка
                                  count(*) as cnt                               -- сколько раз встречается
                           from (
                                   select trim(skill) as skill
                                   from ds_vacancies,
                                        regexp_split_to_table(
                                            coalesce(key_skills, ''),
                                            E'\\t'
                                        ) as skill                               -- разбиваем навыки по табуляции
                                   where key_skills is not null
                                     and key_skills <> ''
                                ) t
                           where skill <> ''
                           group by skill
                           order by cnt desc
                           limit 10                                              -- топ-10 навыков
                        '''

In [113]:
# результат запроса
df = pd.read_sql_query(query_top_skills, connection)
df

,skill,cnt
0,Python,354
1,SQL,208
2,Machine Learning,114
3,Git,81
4,Математическая статистика,62
5,Data Analysis,55
6,Linux,53
7,Pandas,52
8,Data Science,52
9,ML,49


### Выводы по дополнительным исследованиям данных

Анализ распределения зарплат по регионам показал заметную дифференциацию уровней оплаты труда. Более высокие средние и медианные значения наблюдаются в отдельных локациях, однако в большинстве таких регионов количество вакансий невелико. При этом в крупнейших рынках — Москве и Санкт-Петербурге — уровень зарплат ниже экстремальных значений, но отличается большей устойчивостью и репрезентативностью за счёт большего объёма данных. Это позволяет использовать данные по крупным городам в качестве более надёжной основы для рекомендаций.

Сравнение зарплат по уровню опыта подтвердило ожидаемую зависимость дохода от профессионального стажа. Специалисты уровня junior получают существенно меньшую заработную плату по сравнению с более опытными коллегами. Наиболее высокий средний уровень оплаты наблюдается у специалистов уровня middle что отражает высокий спрос на кандидатов с опытом от 3 до 6 лет. Данные по уровню senior представлены ограниченным числом вакансий, поэтому их следует интерпретировать с осторожностью.

Анализ ключевых навыков показал, что рынок вакансий Data Scientist характеризуется чётко выраженным набором базовых требований. Абсолютным лидером по частоте упоминаний является Python за которым следуют SQL и Machine Learning Также значимую роль играют навыки работы с системами контроля версий Git статистикой, анализом данных и специализированными библиотеками Pandas Это формирует типовой профиль Data Scientist и может быть использовано для сопоставления навыков кандидатов с требованиями вакансий.

В целом результаты дополнительных исследований подтверждают, что рынок вакансий Data Scientist структурирован, предъявляет высокие требования к техническим навыкам и демонстрирует устойчивую зависимость уровня заработной платы от опыта работы и региона. Полученные выводы могут быть напрямую использованы для повышения качества фильтрации вакансий, формирования персонализированных рекомендаций и дальнейшего построения прогнозных моделей.

### Прогнозы и варианты продолжения исследования

На основе проведённого анализа данных и выявленных закономерностей можно сформулировать ряд прогнозов и определить направления дальнейшего развития исследования.

С учётом распределения вакансий и уровня заработных плат можно прогнозировать сохранение высокого спроса на специалистов Data Scientist уровня middle поскольку именно для этой группы наблюдается наибольшее количество вакансий и наиболее высокий средний уровень оплаты труда. Вакансии для начинающих специалистов, напротив, останутся ограниченными, что указывает на сохранение высокого порога входа в профессию в ближайшей перспективе.

Анализ ключевых навыков позволяет предположить, что Python** и SQL сохранят статус базовых и обязательных компетенций для Data Scientist. Навыки в области Machine Learning и анализа данных будут оставаться ключевыми факторами, влияющими на уровень заработной платы и востребованность специалистов. Расширение набора навыков в этих областях может рассматриваться как один из основных способов карьерного роста и увеличения дохода.

В качестве логичного продолжения исследования можно выделить построение "прогнозной модели заработной платы" которая будет оценивать ожидаемый уровень дохода специалиста на основе его опыта, набора навыков, региона и формата работы. Такая модель может использоваться как для рекомендаций вакансий, так и для консультаций кандидатов.

Другим направлением развития проекта является создание "рекомендательной системы вакансий", которая будет сопоставлять профиль кандидата (опыт, навыки, предпочтения по формату работы и локации) с характеристиками вакансий. Это позволит автоматически подбирать наиболее релевантные предложения и повысить качество рекомендаций кадрового агентства.

Дополнительно исследование может быть расширено за счёт анализа текстов описаний вакансий с применением методов обработки естественного языка. Это позволит выявлять скрытые требования, уточнять уровень позиции и более точно классифицировать вакансии внутри направления Data Scientist.

Таким образом, дальнейшее развитие исследования предполагает переход от описательного анализа к прогнозированию и автоматизации подбора вакансий, что напрямую соответствует бизнес-задаче кадрового агентства.

In [114]:
connection.close()